# Training all ML models

This notebook:
1. **Recalculates features** for ALL competitions (leagues, cups, European, international)
2. **Includes player data** (rating, goals, assists) as additional features
3. **Trains all models** (LR, RF, XGBoost, LightGBM, KNN, MLP, Ensemble)
4. **Saves models** to `data/models/universal_predictor.pkl`

## Data sources:
- `data/league/` - 19+ domestic leagues
- `data/cups/` - domestic cups (FA Cup, Copa del Rey, etc.)
- `data/european/` - Champions League, Europa League, etc.
- `data/international/` - Euro, World Cup, Nations League

## When to run?
- After fetching new finished matches via `scrape_all.py`
- When you want to retrain models on the latest data

## Workflow:
1. Run `python scrape_all.py` - fetch new matches
2. Run `python regenerate_all_features.py` - recalculate features
3. Run **this notebook** - train models
4. Run `python predict_today.py` - predict today's matches

In [1]:
# Install libraries (one-time)
%pip install scikit-learn xgboost lightgbm pandas numpy joblib psutil matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / 'data'
MODELS_DIR = DATA_DIR / 'models'

COMPETITION_TYPES = ['league', 'cups', 'european', 'international']

print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Competition types: {', '.join(COMPETITION_TYPES)}")

Date: 2026-02-25 18:33
Data directory: e:\Visual Studio\repo\SportWebApp\SofascoreDate\data
Models directory: e:\Visual Studio\repo\SportWebApp\SofascoreDate\data\models
Competition types: league, cups, european, international


## 1. Feature Calculation

Features are variables that models use to predict outcomes:
- Team form (last 5 and 10 matches)
- Table position
- H2H statistics
- Momentum, fatigue, opponent strength
- **NEW:** Player data (rating, goals, assists)

In [3]:
# Reload modules (in case of source code changes)
import importlib
import sofascore.features
importlib.reload(sofascore.features)

from sofascore.features import MLFeatureGenerator

def load_player_stats(comp_dir: Path) -> list:
    player_stats_dir = comp_dir / 'player_stats'
    if not player_stats_dir.exists():
        return []
    
    all_stats = []
    for ps_file in player_stats_dir.glob('*.json'):
        try:
            with open(ps_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_stats.extend(data.get('player_stats', []))
        except:
            continue
    return all_stats

def regenerate_features_for_competition(comp_type: str, country: str, competition: str):
    # European has flat structure: data/european/{competition}/
    if comp_type == 'european':
        comp_dir = DATA_DIR / comp_type / competition
    else:
        comp_dir = DATA_DIR / comp_type / country / competition
    
    raw_dir = comp_dir / 'raw'
    features_dir = comp_dir / 'features'
    
    if not raw_dir.exists():
        return 0
    
    all_matches = []
    for raw_file in sorted(raw_dir.glob('*.json')):
        if 'upcoming' in raw_file.name or 'all_seasons' in raw_file.name:
            continue
        try:
            with open(raw_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            matches = data.get('matches', [])
            all_matches.extend(matches)
        except:
            continue
    
    if not all_matches:
        return 0
    
    player_stats = load_player_stats(comp_dir)
    
    all_matches = sorted(all_matches, key=lambda x: (x.get('date') or '', x.get('round') or 0))
    
    finished_matches = [m for m in all_matches if m.get('home_score') is not None]
    
    if not finished_matches:
        return 0
    
    class DummyDM:
        pass
    fg = MLFeatureGenerator(DummyDM())
    
    elo_table = fg._compute_elo_table(finished_matches)
    
    all_features = []
    for match in finished_matches:
        # Cup matches may not have a round - use 0
        match_round = match.get('round', 0) or 0
        if match_round and match_round < 3:
            continue  # Skip very early rounds (little history)
        
        features = fg.generate_match_features(
            match, finished_matches,
            player_stats=player_stats, elo_table=elo_table
        )
        all_features.append(features)
    
    features_dir.mkdir(exist_ok=True)
    output_file = features_dir / 'features_all_seasons.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump({
            'metadata': {
                'comp_type': comp_type,
                'country': country,
                'competition': competition,
                'total_samples': len(all_features),
                'has_player_stats': len(player_stats) > 0,
                'regenerated_at': datetime.now().isoformat(),
            },
            'samples': all_features
        }, f, ensure_ascii=False, indent=2)
    
    return len(all_features)

print("MLFeatureGenerator module reloaded successfully.")

c:\Users\tomas\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\tomas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


MLFeatureGenerator module reloaded successfully.


In [4]:
print("="*70)
print("RECALCULATING FEATURES FOR ALL COMPETITIONS")
print("="*70)

total_matches = 0
total_competitions = 0
stats_by_type = {}

for comp_type in COMPETITION_TYPES:
    comp_type_dir = DATA_DIR / comp_type
    if not comp_type_dir.exists():
        continue
    
    type_matches = 0
    type_count = 0
    
    print(f"\n[{comp_type.upper()}]")
    
    # European has flat structure (no country subfolder)
    if comp_type == 'european':
        for comp_dir in sorted(comp_type_dir.iterdir()):
            if not comp_dir.is_dir():
                continue
            # Check if this is directly a competition (has raw/ folder)
            if (comp_dir / 'raw').exists():
                country = 'uefa'
                competition = comp_dir.name
                count = regenerate_features_for_competition(comp_type, country, competition)
                if count > 0:
                    print(f"  {competition}: {count} matches")
                    type_matches += count
                    type_count += 1
            else:
                # Could be a country subfolder (like other types)
                for sub_dir in sorted(comp_dir.iterdir()):
                    if not sub_dir.is_dir():
                        continue
                    country = comp_dir.name
                    competition = sub_dir.name
                    count = regenerate_features_for_competition(comp_type, country, competition)
                    if count > 0:
                        print(f"  {country}/{competition}: {count} matches")
                        type_matches += count
                        type_count += 1
    else:
        for country_dir in sorted(comp_type_dir.iterdir()):
            if not country_dir.is_dir():
                continue
            for comp_dir in sorted(country_dir.iterdir()):
                if not comp_dir.is_dir():
                    continue
                
                country = country_dir.name
                competition = comp_dir.name
                
                count = regenerate_features_for_competition(comp_type, country, competition)
                if count > 0:
                    print(f"  {country}/{competition}: {count} matches")
                    type_matches += count
                    type_count += 1
    
    if type_count > 0:
        stats_by_type[comp_type] = {'competitions': type_count, 'matches': type_matches}
        total_matches += type_matches
        total_competitions += type_count

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for comp_type, stats in stats_by_type.items():
    print(f"  {comp_type}: {stats['competitions']} competitions, {stats['matches']} matches")
print(f"\nTOTAL: {total_competitions} competitions, {total_matches} matches")

RECALCULATING FEATURES FOR ALL COMPETITIONS

[LEAGUE]
  austria/bundesliga: 837 matches
  belgium/jupiler_pro_league: 1378 matches
  england/championship: 2497 matches
  england/league_one: 2491 matches
  england/league_two: 2495 matches
  england/premier_league: 1691 matches
  france/ligue_1: 1491 matches
  france/ligue_2: 1567 matches
  germany/2_bundesliga: 1347 matches
  germany/bundesliga: 1345 matches
  greece/super_league: 997 matches
  italy/serie_a: 1681 matches
  italy/serie_b: 1709 matches
  netherlands/eredivisie: 1388 matches
  poland/1_liga: 1341 matches
  poland/ekstraklasa: 1330 matches
  portugal/primeira_liga: 1347 matches
  scotland/premiership: 1036 matches
  spain/la_liga: 1669 matches
  spain/la_liga_2: 2052 matches
  turkey/super_lig: 1527 matches

[CUPS]
  england/community_shield: 3 matches
  england/efl_cup: 220 matches
  england/fa_cup: 322 matches
  france/coupe_de_france: 965 matches
  france/trophee_des_champions: 3 matches
  germany/dfb_pokal: 69 matches


## 2. Training all models

In [5]:
from sofascore.predictor import UniversalPredictor

predictor = UniversalPredictor(str(DATA_DIR))

print("Loading data from ALL competitions...")
print("(leagues, cups, European, international)\n")

df = predictor.load_all_data()  # Loads league, cups, european, international
print(f"\n{'='*50}")
print(f"Total loaded: {len(df)} matches")
print(f"{'='*50}")

Loading data from ALL competitions...
(leagues, cups, European, international)

  Loaded: league/austria/bundesliga (837 matches)
  Loaded: league/belgium/jupiler_pro_league (1378 matches)
  Loaded: league/england/championship (2497 matches)
  Loaded: league/england/league_one (2491 matches)
  Loaded: league/england/league_two (2495 matches)
  Loaded: league/england/premier_league (1691 matches)
  Loaded: league/france/ligue_1 (1491 matches)
  Loaded: league/france/ligue_2 (1567 matches)
  Loaded: league/germany/2_bundesliga (1347 matches)
  Loaded: league/germany/bundesliga (1345 matches)
  Loaded: league/greece/super_league (997 matches)
  Loaded: league/italy/serie_a (1681 matches)
  Loaded: league/italy/serie_b (1709 matches)
  Loaded: league/netherlands/eredivisie (1388 matches)
  Loaded: league/poland/1_liga (1341 matches)
  Loaded: league/poland/ekstraklasa (1330 matches)
  Loaded: league/portugal/primeira_liga (1347 matches)
  Loaded: league/scotland/premiership (1036 matches)


In [6]:
import importlib
import sofascore.predictor
importlib.reload(sofascore.predictor)

# IMPORTANT: After reload, re-import the class and recreate predictor,
# because the old object still uses methods from the old module
from sofascore.predictor import UniversalPredictor, TARGET_CONFIGS
predictor = UniversalPredictor(str(DATA_DIR))

print("\n" + "="*70)
print("MODEL TRAINING")
print("="*70)
print("Classification models: LR, RF, KNN, SVM, MLP, XGBoost, LightGBM, Ensemble, Stacking")
print("Regression models: RF, XGBoost, LightGBM")
print()

# All targets for training
TRAIN_TARGETS = [
    'result', 'btts', 'over_2_5', 'over_1_5',
    'corners_over_8_5', 'corners_over_10_5',
    'cards_over_3_5', 'cards_over_4_5',
    'total_goals', 'total_corners', 'total_cards',
]

print("Classification targets:")
print("  - result (HOME/DRAW/AWAY)")
print("  - btts (Both Teams To Score)")
print("  - over_2_5 / over_1_5 (goals)")
print("  - corners_over_8_5 / corners_over_10_5 (corner kicks)")
print("  - cards_over_3_5 / cards_over_4_5 (cards)")
print()
print("Regression targets:")
print("  - total_goals, total_corners, total_cards")
print()

all_results = predictor.train_all_models(df, targets=TRAIN_TARGETS)

print("\n" + "="*70)
print("TRAINING RESULTS - ALL TARGETS")
print("="*70)

for target, results in all_results.items():
    config = TARGET_CONFIGS.get(target, {})
    is_regression = config.get('task') == 'regression'
    print(f"\n  [{target.upper()}] {'(regression)' if is_regression else '(classification)'}")
    
    if is_regression:
        sorted_results = sorted(results.items(), key=lambda x: x[1])  # MAE ascending
        best_val = sorted_results[0][1] if sorted_results else 0
        for name, mae in sorted_results:
            marker = " <-- best" if mae == best_val else ""
            print(f"    {name:25} MAE={mae:.3f}{marker}")
    else:
        sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
        best_acc = sorted_results[0][1] if sorted_results else 0
        for name, acc in sorted_results:
            marker = " <-- best" if acc == best_acc else ""
            print(f"    {name:25} {acc:.1%}{marker}")

print("\n" + "="*70)
print("TECHNICAL METRICS (training time, model size)")
print("="*70)

for target in all_results:
    stats = predictor.training_stats.get(target, {})
    detailed = stats.get('detailed_metrics', {})
    if not detailed:
        continue
    print(f"\n  [{target.upper()}] ({stats.get('features', 0)} features, "
          f"train={stats.get('train_matches', 0)}, test={stats.get('test_matches', 0)})")
    
    for name, metrics in detailed.items():
        time_s = metrics.get('train_time_s', 0)
        size_kb = metrics.get('model_size_kb', 0)
        if 'accuracy' in metrics:
            print(f"    {name:25} acc={metrics['accuracy']:.1%} f1={metrics.get('f1', 0):.1%} "
                  f"[{time_s:.1f}s, {size_kb:.0f}KB]")
        elif 'mae' in metrics:
            print(f"    {name:25} MAE={metrics['mae']:.3f} R²={metrics.get('r2', 0):.3f} "
                  f"[{time_s:.1f}s, {size_kb:.0f}KB]")


MODEL TRAINING
Classification models: LR, RF, KNN, SVM, MLP, XGBoost, LightGBM, Ensemble, Stacking
Regression models: RF, XGBoost, LightGBM

Classification targets:
  - result (HOME/DRAW/AWAY)
  - btts (Both Teams To Score)
  - over_2_5 / over_1_5 (goals)
  - corners_over_8_5 / corners_over_10_5 (corner kicks)
  - cards_over_3_5 / cards_over_4_5 (cards)

Regression targets:
  - total_goals, total_corners, total_cards


  TRAINING TARGET: RESULT [MULTICLASS]
  Auto-discovered 89 new features beyond reference list

  Dateset: 37317 matches, 230 features
  Class distribution:
    HOME: 16343 (43.8%)
    DRAW: 9235 (24.7%)
    AWAY: 11739 (31.5%)

  Per-competition temporal split (34 competitions):
    Train: 29841, Test: 7476
  Correlation filter: removing 25 features (r > 0.95)
  MI selection: keeping 60 of 205 (threshold=0.005, removed 145)
  Top 5 features (MI): elo_diff, ppg_diff, form10_points_diff, stats_possession_diff, wform_ppg_diff
  Final features: 60

  Training models...
   

In [7]:
MODELS_DIR.mkdir(exist_ok=True)
models_path = MODELS_DIR / 'universal_predictor.pkl'

predictor.save_models(str(models_path))

print(f"\nModels saved to: {models_path}")
print(f"File size: {models_path.stat().st_size / 1024 / 1024:.1f} MB")

Models saved to: e:\Visual Studio\repo\SportWebApp\SofascoreDate\data\models\universal_predictor.pkl

Models saved to: e:\Visual Studio\repo\SportWebApp\SofascoreDate\data\models\universal_predictor.pkl
File size: 783.3 MB


## 3. Verification - prediction test

In [8]:
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', message='X has feature names')

print("\n" + "="*70)
print("PREDICTION TEST - Sample match")
print("="*70)

sample = df.sample(1).iloc[0]

print(f"Match: {sample.get('home_team', 'Home')} vs {sample.get('away_team', 'Away')}")
print(f"Actual result: {sample.get('label_result', '?')}")
print(f"Goals: {sample.get('label_home_goals', 0)} - {sample.get('label_away_goals', 0)}")
print(f"BTTS: {'YES' if sample.get('label_btts') == 1 else 'NO'}")
print(f"Over 2.5: {'YES' if sample.get('label_over_2_5') == 1 else 'NO'}")
if sample.get('label_total_corners') is not None:
    print(f"Corners: {sample.get('label_total_corners', '?')}")
if sample.get('label_total_cards') is not None:
    print(f"Cards: {sample.get('label_total_cards', '?')}")

features = sample.to_dict()
all_preds = predictor.predict_match_all_targets(features)

for target, preds in all_preds.items():
    cons = preds.get('consensus', {})
    if cons.get('task') == 'regression':
        print(f"\n  [{target.upper()}] Prediction: {cons.get('prediction')} "
              f"(range: {cons.get('min')}-{cons.get('max')}, models: {cons.get('n_models')})")
    else:
        print(f"\n  [{target.upper()}] Consensus: {cons.get('prediction')} ({cons.get('agreement')})")
        avg_probs = cons.get('avg_probabilities', {})
        if avg_probs:
            parts = [f"{k}: {v:.1f}%" for k, v in avg_probs.items()]
            print(f"    Average probabilities: {' | '.join(parts)}")


PREDICTION TEST - Sample match
Match: Gillingham vs AFC Wimbledon
Actual result: D
Goals: 0 - 0
BTTS: NIE
Over 2.5: NIE
Corners: 8.0
Cards: 3.0

  [RESULT] Consensus: DRAW (8/8)
    Average probabilities: HOME: 24.2% | DRAW: 52.6% | AWAY: 23.2%

  [BTTS] Consensus: NO (8/8)
    Average probabilities: NO: 64.4% | YES: 35.6%

  [OVER_2_5] Consensus: UNDER (8/8)
    Average probabilities: UNDER: 69.0% | OVER: 31.0%

  [OVER_1_5] Consensus: UNDER (8/8)
    Average probabilities: UNDER: 69.1% | OVER: 30.9%

  [CORNERS_OVER_8_5] Consensus: UNDER (8/8)
    Average probabilities: UNDER: 62.4% | OVER: 37.6%

  [CORNERS_OVER_10_5] Consensus: UNDER (5/8)
    Average probabilities: UNDER: 57.9% | OVER: 42.1%

  [CARDS_OVER_3_5] Consensus: UNDER (8/8)
    Average probabilities: UNDER: 68.2% | OVER: 31.8%

  [CARDS_OVER_4_5] Consensus: UNDER (8/8)
    Average probabilities: UNDER: 72.8% | OVER: 27.2%

  [TOTAL_GOALS] Prediction: 2.12 (range: 1.97-2.23, models: 4)

  [TOTAL_CORNERS] Prediction: 10.0

## 4. Summary

Models were trained on data from:
- **Domestic leagues** (Premier League, La Liga, Serie A, etc.)
- **Cups** (FA Cup, Copa del Rey, DFB-Pokal, etc.)
- **European competitions** (Champions League, Europa League, Conference)
- **International competitions** (Euro, World Cup, Nations League)

**Next steps:**
1. Run `python predict_today.py` for predictions
2. After new matches - run `python scrape_all.py` and `python regenerate_all_features.py`
3. Repeat training every few days/weeks

In [ ]:
from sofascore.predictor import TARGET_CONFIGS

print("="*70)
print("DONE!")
print("="*70)
print(f"\nUpdated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
targets_trained = list(predictor.models.keys())

cls_targets = [t for t in targets_trained if TARGET_CONFIGS.get(t, {}).get('task') != 'regression']
reg_targets = [t for t in targets_trained if TARGET_CONFIGS.get(t, {}).get('task') == 'regression']
total_models = sum(len(m) for m in predictor.models.values())

print(f"Classification: {len(cls_targets)} targets ({', '.join(cls_targets)})")
print(f"Regression: {len(reg_targets)} targets ({', '.join(reg_targets)})")
print(f"Total trained models: {total_models}")
print(f"Saved to: {models_path}")
print(f"\nYou can now run:")
print(f"  python predict_today.py --scrape         predictions")
print(f"  python top_picks/generate_top_picks.py   top picks")